In [2]:
import sys
sys.path.insert(0, '/app')

from database import SessionLocal
from db_models import EventType

db = SessionLocal()
event_types = db.query(EventType).all()
for et in event_types:
    print(f"{et.display_name} - {et.color}")

JCC Sunday - #3b82f6


In [3]:
import sys
sys.path.insert(0, '/app')

from database import SessionLocal
from db_models import UserEventTypeMembership, User, EventType

db = SessionLocal()

# Récupérer tous les memberships avec les infos user et event_type
memberships = db.query(
    UserEventTypeMembership,
    User,
    EventType
).join(
    User, UserEventTypeMembership.user_id == User.id
).join(
    EventType, UserEventTypeMembership.event_type_id == EventType.id
).all()

# Afficher
for membership, user, event_type in memberships:
    print(f"User: {user.display_name} ({user.email})")
    print(f"  Event Type: {event_type.display_name}")
    print(f"  Membership: {membership.membership_type}")
    print(f"  Credits: {membership.remaining_credits}")
    print()

# Ou juste pour un user spécifique
user_id = 1
user_memberships = db.query(
    UserEventTypeMembership,
    EventType
).join(
    EventType
).filter(
    UserEventTypeMembership.user_id == user_id
).all()

print(f"\n--- Memberships for user {user_id} ---")
for membership, event_type in user_memberships:
    print(f"{event_type.display_name}: {membership.membership_type} - {membership.remaining_credits} credits")

User: User1 (user1@example.com)
  Event Type: JCC Sunday
  Membership: full_member
  Credits: None


--- Memberships for user 1 ---


In [4]:
"""
Script to get full members by event type

Usage:
    python get_full_members.py
"""

from sqlalchemy.orm import Session
from database import SessionLocal
from db_models import User, EventType, UserEventTypeMembership


def get_full_members_by_event_type(db: Session):
    """
    Get all full members grouped by event type
    
    Returns:
        dict: {
            'event_type_name': {
                'display_name': str,
                'full_members': [list of emails]
            }
        }
    """
    # Get all event types
    event_types = db.query(EventType).order_by(EventType.id).all()
    
    result = {}
    
    for event_type in event_types:
        # Get all full members for this event type
        full_members = db.query(User).join(
            UserEventTypeMembership,
            UserEventTypeMembership.user_id == User.id
        ).filter(
            UserEventTypeMembership.event_type_id == event_type.id,
            UserEventTypeMembership.membership_type == 'full_member'
        ).order_by(User.email).all()
        
        result[event_type.event_type_name] = {
            'display_name': event_type.display_name,
            'color': event_type.color,
            'full_members': [user.email for user in full_members]
        }
    
    return result


def print_full_members():
    """Print full members by event type in a nice format"""
    db = SessionLocal()
    
    try:
        data = get_full_members_by_event_type(db)
        
        print("\n" + "="*60)
        print("FULL MEMBERS BY EVENT TYPE")
        print("="*60 + "\n")
        
        for event_type_name, info in data.items():
            print(f"📅 {info['display_name']} ({event_type_name})")
            print(f"   Color: {info['color']}")
            print(f"   Full Members: {len(info['full_members'])}")
            
            if info['full_members']:
                print("\n   Members:")
                for email in info['full_members']:
                    print(f"   - {email}")
            else:
                print("   (No full members)")
            
            print("\n" + "-"*60 + "\n")
        
    finally:
        db.close()


def get_full_members_as_csv(event_type_name: str = None):
    """
    Get full members as CSV format (for copy-paste into admin form)
    
    Args:
        event_type_name: Optional - get only for specific event type
    
    Returns:
        str: Comma-separated emails
    """
    db = SessionLocal()
    
    try:
        if event_type_name:
            # Get specific event type
            event_type = db.query(EventType).filter(
                EventType.event_type_name == event_type_name
            ).first()
            
            if not event_type:
                return f"Event type '{event_type_name}' not found"
            
            full_members = db.query(User).join(
                UserEventTypeMembership,
                UserEventTypeMembership.user_id == User.id
            ).filter(
                UserEventTypeMembership.event_type_id == event_type.id,
                UserEventTypeMembership.membership_type == 'full_member'
            ).order_by(User.email).all()
            
            emails = [user.email for user in full_members]
            
            print(f"\n📋 Full members for {event_type.display_name}:")
            print(f"   Total: {len(emails)}")
            print("\n   Copy-paste format:")
            print("   " + ", ".join(emails))
            print("\n   Or line-by-line:")
            for email in emails:
                print(f"   {email}")
            
            return ", ".join(emails)
        
        else:
            # Get all event types
            data = get_full_members_by_event_type(db)
            
            for event_type_name, info in data.items():
                print(f"\n📋 {info['display_name']} ({event_type_name}):")
                print(f"   Total: {len(info['full_members'])}")
                
                if info['full_members']:
                    print("   Copy-paste format:")
                    print("   " + ", ".join(info['full_members']))
                else:
                    print("   (No full members)")
    
    finally:
        db.close()


if __name__ == "__main__":
    print("\n🎾 Apollo - Full Members Report\n")
    
    # Option 1: Print nicely formatted
    print_full_members()
    
    # Option 2: Get as CSV for copy-paste
    print("\n" + "="*60)
    print("CSV FORMAT (for admin form)")
    print("="*60)
    get_full_members_as_csv()
    
    print("\n✅ Done!\n")


🎾 Apollo - Full Members Report


FULL MEMBERS BY EVENT TYPE

📅 JCC Sunday (event_1)
   Color: #3b82f6
   Full Members: 1

   Members:
   - user1@example.com

------------------------------------------------------------


CSV FORMAT (for admin form)

📋 JCC Sunday (event_1):
   Total: 1
   Copy-paste format:
   user1@example.com

✅ Done!



In [2]:
# In Python console or create test file
from database import SessionLocal
from db_models import Event, EventType
from datetime import date

db = SessionLocal()

# Check event types
event_types = db.query(EventType).all()
print("Event Types:")
for et in event_types:
    print(f"  - {et.id}: {et.display_name} ({et.event_type_name})")

# Check events
events = db.query(Event).filter(Event.date >= date.today()).all()
print(f"\nFuture Events: {len(events)}")
for e in events:
    et = db.query(EventType).filter(EventType.id == e.event_type_id).first()
    print(f"  - {e.date} | {et.display_name if et else 'Unknown'} | ID={e.id}")

db.close()

Event Types:
  - 2: JCC Sunday (event_4)

Future Events: 2
  - 2025-12-27 | JCC Sunday | ID=2
  - 2025-12-29 | JCC Sunday | ID=3


In [3]:
from database import SessionLocal
from db_models import User, UserEventTypeMembership, EventType

db = SessionLocal()

# Ton email admin
admin_email = "respectmyprivacy007@gmail.com"  # Change si besoin

user = db.query(User).filter(User.email == admin_email).first()
print(f"User ID: {user.id}")

memberships = db.query(UserEventTypeMembership).filter(
    UserEventTypeMembership.user_id == user.id
).all()

print(f"\nMemberships: {len(memberships)}")
for m in memberships:
    et = db.query(EventType).filter(EventType.id == m.event_type_id).first()
    print(f"  - {et.display_name}: {m.membership_type}")

db.close()

User ID: 1

Memberships: 0


In [1]:
from database import SessionLocal
from db_models import Event, EventType, User, UserEventTypeMembership
from datetime import date

db = SessionLocal()

# Ton user email
user_email = "ton_email@example.com"  # Change

user = db.query(User).filter(User.email == user_email).first()
print(f"User: {user.display_name} (ID: {user.id})")

# Ses memberships
memberships = db.query(UserEventTypeMembership, EventType).join(
    EventType, UserEventTypeMembership.event_type_id == EventType.id
).filter(UserEventTypeMembership.user_id == user.id).all()

print("\nMemberships:")
for m, et in memberships:
    print(f"  - {et.display_name}: {m.membership_type}")
    
    # Events pour ce type
    events = db.query(Event).filter(
        Event.event_type_id == et.id,
        Event.date >= date.today()
    ).count()
    print(f"    → {events} future events")

db.close()

AttributeError: 'NoneType' object has no attribute 'display_name'

In [7]:

from database import SessionLocal
from db_models import Event, EventType
from datetime import date

db = SessionLocal()

# All event types
print('=== EVENT TYPES ===')
event_types = db.query(EventType).all()
for et in event_types:
    print(f'{et.id}: {et.event_type_name} - {et.display_name} - {et.color}')

print('\n=== EVENTS ===')
today = date.today()
events = db.query(Event).filter(Event.date >= today).order_by(Event.date).all()
print(f'Total future events: {len(events)}')

for event in events:
    et = db.query(EventType).filter(EventType.id == event.event_type_id).first()
# Après (correct)
    display = et.display_name if et else "NO TYPE"
    print(f'{event.id}: {event.date} - {display} (type_id={event.event_type_id})')
db.close()


=== EVENT TYPES ===
1: event_1 - JCC Sunday - #3b82f6

=== EVENTS ===
Total future events: 1
1: 2025-12-27 - JCC Sunday (type_id=1)


In [ ]:
# Check events in DB
docker exec apollo-app python3 -c "
from database import SessionLocal
from db_models import Event, EventType
from datetime import date

db = SessionLocal()

# All event types
print('=== EVENT TYPES ===')
event_types = db.query(EventType).all()
for et in event_types:
    print(f'{et.id}: {et.event_type_name} - {et.display_name} - {et.color}')

print('\n=== EVENTS ===')
today = date.today()
events = db.query(Event).filter(Event.date >= today).order_by(Event.date).all()
print(f'Total future events: {len(events)}')

for event in events:
    et = db.query(EventType).filter(EventType.id == event.event_type_id).first()
    print(f'{event.id}: {event.date} - {et.display_name if et else \"NO TYPE\"} (type_id={event.event_type_id})')

db.close()
"

In [8]:
from database import SessionLocal
from db_models import User, UserEventTypeMembership, EventType

db = SessionLocal()

# Trouve ton user
user_email = "user1@example.com"  # Change ça!
user = db.query(User).filter(User.email == user_email).first()

if user:
    print(f'=== USER: {user.email} ===')
    print(f'ID: {user.id}')
    print(f'Display name: {user.display_name}')
    
    print('\n=== MEMBERSHIPS ===')
    memberships = db.query(UserEventTypeMembership).filter(
        UserEventTypeMembership.user_id == user.id
    ).all()
    
    for m in memberships:
        et = db.query(EventType).filter(EventType.id == m.event_type_id).first()
        print(f'Event type: {et.event_type_name} ({et.display_name})')
        print(f'  Type: {m.membership_type}')
        print(f'  Credits: {m.remaining_credits}')
else:
    print(f'User {user_email} not found!')

db.close()

=== USER: user1@example.com ===
ID: 3
Display name: User1

=== MEMBERSHIPS ===
Event type: event_1 (JCC Sunday)
  Type: full_member
  Credits: None


In [9]:
from database import SessionLocal
from services.event_service import EventService

db = SessionLocal()

user_id = 3  # user1@example.com

event_service = EventService(db)
events = event_service.get_events_for_schedule(user_id)

print(f'=== EVENTS FOR USER {user_id} ===')
print(f'Total events returned: {len(events)}')

for event in events:
    print(f'\nEvent ID: {event["id"]}')
    print(f'Date: {event["date"]}')
    print(f'Event type ID: {event["event_type_id"]}')
    print(f'Event type name: {event.get("event_type_name")}')
    print(f'User status: {event.get("user_status")}')

db.close()

=== EVENTS FOR USER 3 ===
Total events returned: 1

Event ID: 1
Date: 2025-12-27
Event type ID: 1
Event type name: event_1
User status: None


In [10]:
from database import SessionLocal
from services.event_service import EventService

db = SessionLocal()

user_id = 3

# Get memberships
event_service = EventService(db)
memberships = event_service.get_user_memberships_formatted(user_id)

print('=== MEMBERSHIPS ===')
for key, membership in memberships.items():
    print(f'{key}: {membership}')

# Get events
events = event_service.get_events_for_schedule(user_id)

print(f'\n=== EVENT MATCHING ===')
for event in events:
    event_type_id = event['event_type_id']
    print(f'\nEvent {event["id"]} has event_type_id={event_type_id}')
    
    # Find matching membership
    found = False
    for key, membership in memberships.items():
        if membership['id'] == event_type_id:
            print(f'  ✓ MATCH with {key}: {membership["display_name"]}')
            found = True
            break
    
    if not found:
        print(f'  ✗ NO MATCH - Event will be HIDDEN!')

db.close()

=== MEMBERSHIPS ===
event_1: {'id': 1, 'event_type_name': 'event_1', 'display_name': 'JCC Sunday', 'location': 'JCC Calgary', 'time_start': '15:00', 'time_end': '17:00', 'type': 'full_member', 'remaining_credits': None, 'color': '#3b82f6'}

=== EVENT MATCHING ===

Event 1 has event_type_id=1
  ✓ MATCH with event_1: JCC Sunday


In [2]:
!pip install pandas --break-system-packages

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 3.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 5.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 5.7 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [3]:
# list_and_export_tables.py
from sqlalchemy import inspect
from database import engine, SessionLocal
import pandas as pd
import os

def export_tables_to_csv():
    """List all tables and export first 15 rows of each to CSV"""
    
    # Create output directory
    os.makedirs('table_exports2', exist_ok=True)
    
    # Get all table names
    inspector = inspect(engine)
    tables = inspector.get_table_names()
    
    print(f"Found {len(tables)} tables:")
    for table in tables:
        print(f"  - {table}")
    
    # Export each table
    db = SessionLocal()
    try:
        for table in tables:
            query = f"SELECT * FROM {table} LIMIT 15"
            df = pd.read_sql(query, db.bind)
            
            output_file = f'table_exports2/{table}.csv'
            df.to_csv(output_file, index=False)
            print(f"✓ Exported {table}: {len(df)} rows → {output_file}")
    
    finally:
        db.close()

if __name__ == "__main__":
    export_tables_to_csv()

Found 10 tables:
  - users
  - user_event_type_memberships
  - event_types
  - events
  - password_resets
  - admins
  - messages
  - friends
  - attendees
  - comments
✓ Exported users: 2 rows → table_exports2/users.csv
✓ Exported user_event_type_memberships: 0 rows → table_exports2/user_event_type_memberships.csv
✓ Exported event_types: 1 rows → table_exports2/event_types.csv
✓ Exported events: 2 rows → table_exports2/events.csv
✓ Exported password_resets: 0 rows → table_exports2/password_resets.csv
✓ Exported admins: 2 rows → table_exports2/admins.csv
✓ Exported messages: 0 rows → table_exports2/messages.csv
✓ Exported friends: 0 rows → table_exports2/friends.csv
✓ Exported attendees: 0 rows → table_exports2/attendees.csv
✓ Exported comments: 0 rows → table_exports2/comments.csv
